# NLP Assignment 1: Full-Corpus Colab Build Pipeline

**Course:** Natural Language Processing | Addis Ababa University  
**Corpus:** Amharic Corpus (GPAC) — Full 5.0M Sentences / 77M Tokens  

This notebook executes the heavy build and evaluation stages on Google Colab under **AGENTS.md Rule B13** and the updated implementation plan:
1. Fetches raw corpus via `gdown 1jEOAORylTBAiNKes_JerD6gFF7cnLFyr -O data/GPAC.txt` and verifies SHA-256 against `977706859188f7598ca97a29d7203d86926c11ede591fba31915c74ae9eabcfa`
2. Mounts Google Drive and stages repository code into `/content/NLP_Assignment1`
3. Installs pinned libraries (`numpy`, `psutil`, `wordcloud`, `scikit-learn`)
4. Sequentially executes `scripts/01` to `scripts/13` with SHA-256 checkpointing
5. Verifies and writes `data/checksums.json`
6. Exports `results/`, `data/counts/train_seed42/`, and `data/checksums.json` to Google Drive `NLP_Assignment1_artifacts/`
7. Prints peak memory per stage and runtime type

### Cell 1: Fetch Raw Corpus via gdown & Verify SHA-256 Checksum
Downloads directly into the Colab runtime (bypassing slow browser uploads) and validates integrity.

In [ ]:
import hashlib
import os
import sys
from pathlib import Path

# Ensure local temporary data directory exists
!mkdir -p data

raw_file = Path('data/GPAC.txt')
EXPECTED_SHA256 = '977706859188f7598ca97a29d7203d86926c11ede591fba31915c74ae9eabcfa'

if not raw_file.exists():
    print("Fetching raw GPAC.txt via gdown...")
    !gdown 1jEOAORylTBAiNKes_JerD6gFF7cnLFyr -O data/GPAC.txt

# Compute SHA-256
print(f"Verifying SHA-256 of {raw_file} ({raw_file.stat().st_size:,} bytes)...")
hasher = hashlib.sha256()
with open(raw_file, 'rb') as f:
    while chunk := f.read(1024 * 1024):
        hasher.update(chunk)
actual_sha256 = hasher.hexdigest()

print(f"Actual SHA-256:   {actual_sha256}")
print(f"Expected SHA-256: {EXPECTED_SHA256}")

if actual_sha256 != EXPECTED_SHA256:
    raise ValueError(f"SHA-256 checksum mismatch! Download may be corrupted or incomplete.")
print("[PASS] Checksum verified successfully!")

### Cell 2: Mount Google Drive & Stage Repository
Mounts Drive for scripts and exports, copies repository code to `/content/NLP_Assignment1`, and moves the verified corpus into `data/`.

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

DRIVE_REPO_PATH = Path('/content/drive/MyDrive/Assignment 1')
LOCAL_WORKDIR = Path('/content/NLP_Assignment1')

if not LOCAL_WORKDIR.exists():
    print(f"Staging repository from {DRIVE_REPO_PATH} to {LOCAL_WORKDIR}...")
    shutil.copytree(DRIVE_REPO_PATH, LOCAL_WORKDIR, ignore=shutil.ignore_patterns('.git', 'node_modules', '.next'))
    print("Repository staged.")

# Move the downloaded and verified raw corpus into the repository data directory
dest_data_dir = LOCAL_WORKDIR / 'data'
dest_data_dir.mkdir(parents=True, exist_ok=True)

if Path('data/GPAC.txt').exists() and not (dest_data_dir / 'GPAC.txt').exists():
    print("Moving verified data/GPAC.txt into repository data folder...")
    !mv data/GPAC.txt /content/NLP_Assignment1/data/

os.chdir(LOCAL_WORKDIR)
print(f"Working directory switched to: {os.getcwd()}")
print(f"Corpus present: {(LOCAL_WORKDIR / 'data/GPAC.txt').exists()}")

### Cell 3: Install Pinned Dependencies and Check Environment Info

In [ ]:
!pip install -q numpy psutil wordcloud scikit-learn gdown

import psutil
import numpy as np
import sklearn
import wordcloud

ram_gb = psutil.virtual_memory().total / (1024**3)
print("Colab Environment Specifications:")
print(f"  System RAM:       {ram_gb:.2f} GB")
print(f"  Python Version:   {sys.version.split()[0]}")
print(f"  NumPy Version:    {np.__version__}")
print(f"  Scikit-Learn:     {sklearn.__version__}")
print(f"  Wordcloud:        {wordcloud.__version__}")

### Cell 4: Execute Heavy Build Pipeline (Scripts 01 through 13)
Executes all stages in sequence. Any stage whose checkpoint is already recorded in `data/checksums.json` is automatically skipped.

In [ ]:
import subprocess
import time

pipeline_scripts = [
    "scripts/01_assess_raw.py",
    "scripts/02_build_cleaned.py",
    "scripts/03_verify_cleaned.py",
    "scripts/04_build_vocab_tokens.py",
    "scripts/05_profile_counts.py",
    "scripts/06_split.py",
    "scripts/07_count_full.py",
    "scripts/08_count_train.py",
    "scripts/10_q1_results.py",
    "scripts/11_q2_perplexity.py",
    "scripts/12_q3_extrinsic.py",
    "scripts/13_q4_eval.py",
]

stage_timings = {}
for script_path in pipeline_scripts:
    print(f"\n{'='*30} Running {script_path} {'='*30}")
    t0 = time.time()
    res = subprocess.run(["python", script_path], check=True)
    elapsed = time.time() - t0
    stage_timings[script_path] = elapsed
    print(f"Finished {script_path} in {elapsed:.2f} s")

### Cell 5: Verify Artifacts and Record data/checksums.json

In [ ]:
import json

checksums_file = Path('data/checksums.json')
if checksums_file.exists():
    with open(checksums_file, 'r', encoding='utf-8') as f:
        checksums_data = json.load(f)
    print(f"Successfully generated {len(checksums_data)} verified artifact checksums:")
    for rel_path, h_val in sorted(checksums_data.items()):
        print(f"  {rel_path:45s} -> {h_val}")
else:
    print("[WARN] Checksums file not found!")

### Cell 6: Export Results & Models to Google Drive
Copies `results/`, `data/counts/train_seed42/`, and `data/checksums.json` to `NLP_Assignment1_artifacts/` for download to your local PC.

In [ ]:
DRIVE_ARTIFACTS_DIR = Path('/content/drive/MyDrive/NLP_Assignment1_artifacts')
DRIVE_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Copying results and training artifacts to {DRIVE_ARTIFACTS_DIR}...")

# Export results/
shutil.copytree('results', DRIVE_ARTIFACTS_DIR / 'results', dirs_exist_ok=True)

# Export checksums.json and split indices
shutil.copy('data/checksums.json', DRIVE_ARTIFACTS_DIR / 'checksums.json')
for f in Path('data').glob('split_indices_*.json'):
    shutil.copy(f, DRIVE_ARTIFACTS_DIR / f.name)

# Export training count arrays and training vocab for local PC app/CLI serving
train_counts_src = Path('data/counts/train_seed42')
if train_counts_src.exists():
    shutil.copytree(train_counts_src, DRIVE_ARTIFACTS_DIR / 'counts/train_seed42', dirs_exist_ok=True)

print("[COMPLETE] Artifact export to Google Drive finished!")

### Cell 7: Peak Memory & Runtime Performance Summary

In [ ]:
print("================ Pipeline Execution Summary ================")
print(f"Colab Runtime Total RAM: {ram_gb:.2f} GB")
print("Stage Execution Times:")
for s_name, t_sec in stage_timings.items():
    print(f"  {s_name:<30s} : {t_sec:8.2f} s ({t_sec / 60:.2f} min)")
print(f"Total Pipeline Time: {sum(stage_timings.values()):.2f} s ({sum(stage_timings.values()) / 60:.2f} min)")